In [ ]:
# 1) GPU check + package installs
import sys
print(sys.version)

!nvidia-smi
!{sys.executable} -m pip install -q pretty_midi mido tqdm

In [ ]:
# 2) Imports + configuration
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import torch

SEED = 42
VALIDATION_SPLIT = 0.2
TOKEN_SEQUENCE_LENGTH = 512
WINDOW_STEP = 128

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CODE_INPUT_CANDIDATES = [
    Path('/kaggle/input/music-generation-unsupervised-task4-src'),
    Path('/kaggle/input/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/music-generation-unsupervised-task2-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task4-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task2-src'),
]

code_input_root = None
for candidate in CODE_INPUT_CANDIDATES:
    if candidate.exists():
        code_input_root = candidate
        break

if code_input_root is None:
    raise FileNotFoundError(
        'Code dataset path not found. Run "!ls /kaggle/input" and update CODE_INPUT_CANDIDATES.'
    )

if (code_input_root / 'music-generation-unsupervised').exists():
    code_input_root = code_input_root / 'music-generation-unsupervised'

PROJECT_ROOT = Path('/kaggle/working/music-generation-unsupervised')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(code_input_root, PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Code input used:', code_input_root)
print('Project root exists:', PROJECT_ROOT.exists())

In [ ]:
# 3) Data loading and split preparation
from src.preprocessing.midi_parser import discover_midi_files
from src.preprocessing.split_manager import get_or_create_train_val_split

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/jackvial/themaestrodatasetv2'),
    Path('/kaggle/input/lakh-midi-clean'),
    Path('/kaggle/input/lakh-midi-dataset'),
    Path('/kaggle/input/lmd-full'),
    Path('/kaggle/input/lmd-matched'),
]

DATA_ROOT = None
for candidate in DATASET_CANDIDATES:
    if candidate.exists():
        DATA_ROOT = candidate
        break

if DATA_ROOT is None:
    raise FileNotFoundError('No MIDI dataset found in DATASET_CANDIDATES.')

midi_files = discover_midi_files(DATA_ROOT)
if not midi_files:
    raise ValueError(f'No MIDI files found under: {DATA_ROOT}')

train_files, validation_files = get_or_create_train_val_split(
    midi_files=midi_files,
    split_name='preprocessing',
    validation_split=VALIDATION_SPLIT,
    seed=SEED,
)

print('Data root:', DATA_ROOT)
print('Total MIDI files:', len(midi_files))
print('Train files:', len(train_files))
print('Validation files:', len(validation_files))

In [ ]:
# 4) Build token chunks and piano-roll windows
from src.preprocessing.piano_roll import load_windowed_piano_rolls
from src.preprocessing.tokenizer import build_token_chunks_from_files

train_chunks = build_token_chunks_from_files(train_files, chunk_length=TOKEN_SEQUENCE_LENGTH)
val_chunks = build_token_chunks_from_files(validation_files, chunk_length=TOKEN_SEQUENCE_LENGTH)

train_windows = load_windowed_piano_rolls(
    midi_files=train_files,
    sequence_length=TOKEN_SEQUENCE_LENGTH,
    step_size=WINDOW_STEP,
)
val_windows = load_windowed_piano_rolls(
    midi_files=validation_files,
    sequence_length=TOKEN_SEQUENCE_LENGTH,
    step_size=WINDOW_STEP,
)

print('Train token chunks:', len(train_chunks))
print('Validation token chunks:', len(val_chunks))
print('Train piano-roll windows:', train_windows.shape[0])
print('Validation piano-roll windows:', val_windows.shape[0])

In [ ]:
# 5) Summary
print('Preprocessing completed successfully.')
print('You can now use these splits/windows/chunks in training notebooks.')